In [18]:
import struct
from pathlib import Path

EXPECTED_HEADER = 0xA3
HEADER_SIZE = 3
FIXED_FIELDS_SIZE = 14   # counter(4) + timestamp(8) + num_samples(2)
MIN_PAYLOAD_LEN = FIXED_FIELDS_SIZE


def load_binary_file(path: str) -> bytes:
    return Path(path).read_bytes()


def parse_one_packet(buf: bytes, offset: int):
    if offset + HEADER_SIZE > len(buf):
        raise ValueError("Not enough bytes for header")

    header = buf[offset]
    if header != EXPECTED_HEADER:
        raise ValueError(f"Bad header at offset {offset}: 0x{header:02X}")

    payload_len = (buf[offset + 1] << 8) | buf[offset + 2]
    total_size = HEADER_SIZE + payload_len

    if payload_len < MIN_PAYLOAD_LEN:
        raise ValueError(
            f"Invalid payload_len={payload_len} at offset {offset}"
        )

    if offset + total_size > len(buf):
        raise ValueError(
            f"Incomplete packet at offset {offset}: need {total_size} bytes"
        )

    pkt = buf[offset:offset + total_size]

    counter = struct.unpack_from("<I", pkt, 3)[0]
    timestamp = struct.unpack_from("<Q", pkt, 7)[0]
    num_samples = struct.unpack_from("<H", pkt, 15)[0]
    biogap_payload = pkt[17:]

    return {
        "offset": offset,
        "header": header,
        "payload_len": payload_len,
        "total_size": total_size,
        "counter": counter,
        "timestamp_us": timestamp,
        "num_samples": num_samples,
        "biogap_payload": biogap_payload,
        "biogap_payload_len": len(biogap_payload),
    }, offset + total_size


def parse_stream(buf: bytes):
    packets = []
    offset = 0

    while offset < len(buf):
        pkt, offset = parse_one_packet(buf, offset)
        packets.append(pkt)

    return packets


def print_summary(packets, preview_len=16):
    for i, pkt in enumerate(packets):
        preview = pkt["biogap_payload"][:preview_len].hex(" ")
        print(
            f"Packet {i:04d} | "
            f"off={pkt['offset']:6d} | "
            f"header=0x{pkt['header']:02X} | "
            f"plen={pkt['payload_len']:4d} | "
            f"size={pkt['total_size']:4d} | "
            f"cnt={pkt['counter']:10d} | "
            f"ts={pkt['timestamp_us']:12d} | "
            f"nsamp={pkt['num_samples']:5d} | "
            f"bio_len={pkt['biogap_payload_len']:4d} | "
            f"payload[:{preview_len}]={preview}"
        )

    


In [22]:
from pathlib import Path
txt_file_path = Path("D:/DUMMY.TXT")
raw = load_binary_file(txt_file_path)
packets = parse_stream(raw)

In [ ]:
timestamps = [pkt["timestamp_us"] for pkt in packets]

np.diff(timestamps)


[553449,
 563739,
 573739,
 583739,
 593739,
 603739,
 613739,
 623739,
 633739,
 643739,
 653739,
 663739,
 673739,
 683739,
 693739,
 703739,
 713739,
 723739,
 733739,
 743739,
 753739,
 763739,
 773739,
 783739,
 793740,
 803739,
 813739,
 823739,
 833739,
 843739,
 853739,
 863739,
 873740,
 883739,
 893740,
 903739,
 913739,
 923739,
 933739,
 943739,
 953739,
 963739,
 973739,
 983740,
 993739,
 1003739,
 1013739,
 1023739,
 1033739,
 1043739,
 1053739,
 1063739,
 1073124,
 1083125,
 1093123,
 1103739,
 1113739,
 1123739,
 1133739,
 1143739,
 1153739,
 1163740,
 1173739,
 1183739,
 1193739,
 1203739,
 1213740,
 1223739,
 1233740,
 1243739,
 1253739,
 1263739,
 1273739,
 1283739,
 1293739,
 1303739,
 1313739,
 1323739,
 1333739,
 1343739,
 1353739,
 1363739,
 1373739,
 1383739,
 1393739,
 1403739,
 1413739,
 1423740,
 1433739,
 1443739,
 1453739,
 1463739,
 1473739,
 1483739,
 1493739,
 1503739,
 1513739,
 1523739,
 1533739,
 1543739,
 1553740,
 1563739,
 1573739,
 1583739,
 1593